In [1]:
import pandas as pd
import os
import gc
import numpy as np
from sklearn.preprocessing import LabelEncoder

### 세그먼트 A, B를 위한 피쳐생성, 그후 생성한 피쳐와 원래 피쳐의 상관계수 구한 후 저장.

In [29]:
train_path='data/train/parquet'

df_list=[]

base_path="data/train/parquet/1_회원정보_train.parquet"
base_df=pd.read_parquet(base_path)
base_df=base_df[['Segment','ID','기준년월']]

# 상관계수를 저장할 딕셔너리 초기화
a_correlations = {}
b_correlations = {}
ab_correlations = {}
for x in os.listdir(train_path):
    
    file_path=os.path.join(train_path,x)
    
    df=pd.read_parquet(file_path)
    df=df.dropna(axis=1)

    # merge 때의 오류를 피하기 위한 if
    if 'Segment' in df.columns:
        pass
    else:    
        df=pd.merge(left=base_df,right=df,on=['ID','기준년월'])
   
    # 피쳐 생성 
    df['ab']= (df['Segment'] == 'A' )| (df['Segment'] == 'B')
    df['a']= df['Segment'] == 'A' 
    df['b']= df['Segment'] == 'B'

     #'ID','기준년월' 칼럼 제거
    df=df.drop(['ID','기준년월'],axis=1)
    # 고유 값이 1개인 컬럼들 찾아서 삭제(상관계수 못구하니가)
    cols_to_drop = [col for col in df.columns if df[col].nunique() == 1]
    df = df.drop(columns=cols_to_drop)
    
    # LabelEncoder 하기
    le = LabelEncoder()
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = le.fit_transform(df[col].astype(str))
        
   
    # 각 열과 Target_Column 간의 상관계수 계산
    for target in ['a','b','ab']:
        other_numeric_cols=df.drop(target, errors='ignore')
        target_series = df[target]        
        
        
        for col in other_numeric_cols:
            eval(f'{target}_correlations')[col] = target_series.corr(df[col])
    
        gc.collect()
    print(x,'   완료')


1_회원정보_train.parquet    완료
2_신용정보_train.parquet    완료
3_승인매출정보_train.parquet    완료
4_청구입금정보_train.parquet    완료
5_잔액정보_train.parquet    완료
6_채널정보_train.parquet    완료
7_마케팅정보_train.parquet    완료
8_성과정보_train.parquet    완료


In [67]:
#상관관계 딕셔너리들을 데이터 프레임으로 변환 후 저장
corrs=[pd.Series(eval(f'{target}_correlations'))  for target in ['a','b','ab']]
corr_df=pd.concat(corrs,axis=1)
corr_df.columns=['a','b','ab'] 

corr_df = corr_df.loc[(corr_df.abs().sum(axis=1)).sort_values(ascending=False).index]
corr_df.to_csv("data/abcorr.csv",encoding='utf-8-sig') 


### Segment로 구한 상관계수와 ab 에 집중해 구한 상관계수 비교

In [85]:
#비교를 위한 Segment와의 상관계수를 구한 데이터 불러오기
cor=pd.read_csv('data/모든_상관계수_결측치_제거.csv')

idx=cor[cor['Correlation'].abs()>0.4].index
cor=cor.loc[idx]
feature_segment=set(cor['Variable'].values)

In [91]:
# ab 여부를 잘 설명하는 상위 20개 피쳐들
feature_ab=set(corr_df.index[:24])
#ab를 더 잘 설명하는 피쳐를 찾는다.
feature_ab - feature_segment

{'Segment',
 'a',
 'ab',
 'b',
 '이용금액_할부_R12M',
 '이용금액_할부_무이자_R12M',
 '평잔_일시불_해외_6M',
 '포인트_마일리지_환산_B0M',
 '할부금액_3M_R12M',
 '할부금액_무이자_3M_R12M'}

### 작업을 위해 저장

In [89]:
train_path='data/train/parquet'

df_list=[]
for x in os.listdir(train_path):
    file_path=os.path.join(train_path,x)
    
    df=pd.read_parquet(file_path)
    
    #원하는 칼럼만 가져옴
    df=df[list(set(df.columns) & check_col|{'ID','기준년월'})]
    print(df.columns)
    
    
    gc.collect()
    
    df_list.append(df)
df=pd.merge(left=df_list[0],right=df_list[1],on=['ID','기준년월'])
for x in range(7):
    df=pd.merge(left=df,right=df_list[x+1],on=['ID','기준년월'])
    gc.collect()
#'ID','기준년월' 칼럼 제거
df=df.drop(['ID','기준년월'],axis=1)
df.to_csv('data/abfeature.csv',encoding='utf-8-sig',index=False)

Index(['기준년월', 'ID', '_1순위카드이용금액', 'Segment'], dtype='object')
Index(['ID', '기준년월'], dtype='object')
Index(['_1순위업종_이용금액', '정상청구원금_B5M', '이용금액_할부_무이자_R12M', '할부금액_3M_R12M',
       '정상입금원금_B2M', '정상청구원금_B2M', '정상청구원금_B0M', '쇼핑_도소매_이용금액', '정상입금원금_B0M',
       '할부금액_무이자_3M_R12M', 'ID', '기준년월', '정상입금원금_B5M', '이용금액_일시불_R12M',
       '이용금액_오프라인_R6M', '이용금액_할부_R12M'],
      dtype='object')
Index(['청구금액_R6M', 'ID', '기준년월', '포인트_마일리지_환산_B0M', '청구금액_R3M', '청구금액_B0'], dtype='object')
Index(['ID', '평잔_일시불_해외_6M', '기준년월'], dtype='object')
Index(['ID', '기준년월'], dtype='object')
Index(['ID', '기준년월'], dtype='object')
Index(['ID', '기준년월'], dtype='object')
